# Friction — sliding block on a rigid floor

A flat soft block rests on the floor under gravity; its top face is dragged
tangentially (+x). Coulomb friction at the floor resists.

* **FEM (FEniCSx)** uses a penalty-regularised Coulomb law (return mapping) in pure
  UFL and exposes the **friction force** and the **dissipated work** directly. The
  normal load is gravity, so `N = W = rho*g*V` and the friction force plateaus at the
  analytic Coulomb limit `mu*W` — an analytic check on the whole contact model.
* **Newton (XPBD)** sets `soft_contact_mu` and enforces friction as a positional
  projection — it exposes the kinematic **slip** but **no calibrated friction force**,
  exactly as it exposes no calibrated normal contact force. That is the difference.

Data: `python -m fenics_run.run_friction` and `python -m newton_run.run_friction`.

In [ ]:
%matplotlib inline
import os
import numpy as np
import matplotlib.pyplot as plt
from common import params

fem = np.load(params.FEM_FRICTION_NPZ) if os.path.exists(params.FEM_FRICTION_NPZ) else None
nw = np.load(params.NEWTON_FRICTION_NPZ) if os.path.exists(params.NEWTON_FRICTION_NPZ) else None
print('FEM friction:', fem is not None, '| Newton friction:', nw is not None)
print(f'analytic: W = {params.friction_block_weight():.2f} N, mu*W = {params.coulomb_plateau():.2f} N')

## 1. Friction force vs drag — stick, then the Coulomb plateau (FEM)

The tangential force rises while the contact **sticks** (elastic shear), then
saturates at `mu*N` once it **slips**. With the gravity normal load, the plateau
should match the analytic `mu*W`, and the measured normal force `N` should match `W`.

In [ ]:
if fem is not None:
    d = fem['drag'] * 1000.0
    plt.figure(figsize=(6, 5))
    plt.plot(d, fem['friction_force'], 'o-', color='tab:blue', label='FEM friction force')
    plt.plot(d, fem['normal_force'], '^-', color='tab:green', alpha=0.7, label='FEM normal force N')
    plt.axhline(float(fem['plateau']), color='k', ls='--', lw=1.5, label='analytic mu*W')
    plt.axhline(float(fem['weight']), color='grey', ls=':', lw=1.2, label='weight W')
    plt.xlabel('top drag [mm]'); plt.ylabel('force [N]'); plt.legend(); plt.grid(alpha=0.3)
    plt.title('Friction: stick -> slip plateau (FEM vs Coulomb)'); plt.show()
    print(f"peak friction force = {fem['friction_force'].max():.2f} N vs analytic mu*W = {float(fem['plateau']):.2f} N")
    print(f"normal force N = {fem['normal_force'][-1]:.2f} N vs weight W = {float(fem['weight']):.2f} N")
else:
    print('No FEM friction data. Run: python -m fenics_run.run_friction')

## 2. Dissipated frictional work & the stick→slip transition (FEM)

Cumulative work done against friction (dissipated once in steady slip), and the
fraction of the contact area that is slipping (0 = fully stuck, 1 = fully sliding).

In [ ]:
if fem is not None:
    d = fem['drag'] * 1000.0
    fig, ax1 = plt.subplots(figsize=(6, 4))
    ax1.plot(d, fem['friction_work'], 'o-', color='tab:red')
    ax1.set_xlabel('top drag [mm]'); ax1.set_ylabel('cumulative friction work [J]', color='tab:red')
    ax2 = ax1.twinx()
    ax2.plot(d, fem['slip_fraction'], 's--', color='tab:purple', alpha=0.7)
    ax2.set_ylabel('slipping area fraction', color='tab:purple'); ax2.set_ylim(-0.05, 1.05)
    plt.title('Friction: dissipated work & stick/slip (FEM)'); plt.tight_layout(); plt.show()

## 3. The shared axis — bottom slip vs drag (FEM and Newton)

Both solvers expose the kinematics: how far the bottom face slides as the top is
dragged. Below the knee the block sticks (bottom barely moves, block shears); above
it the bottom slides (slip tracks drag). XPBD gives this curve — but, unlike FEM, no
friction force to put on the y-axis of §1.

In [ ]:
plt.figure(figsize=(6, 5))
if fem is not None:
    plt.plot(fem['drag'] * 1000.0, fem['mean_slip'] * 1000.0, 'o-', color='tab:blue', label='FEM mean bottom slip')
if nw is not None:
    plt.plot(nw['drag'] * 1000.0, nw['bottom_slip'] * 1000.0, 's-', color='tab:orange', label='Newton XPBD bottom slip')
mx = params.FRICTION_DRAG_MAX * 1000.0
plt.plot([0, mx], [0, mx], 'k:', lw=1, label='full slip (slip = drag)')
plt.xlabel('top drag [mm]'); plt.ylabel('mean bottom slip [mm]'); plt.legend(); plt.grid(alpha=0.3)
plt.title('Friction: bottom slip vs drag'); plt.show()

## Takeaway

* The FEM reproduces textbook Coulomb friction: a stick branch, a slip plateau at
  `mu*W`, a normal reaction equal to the weight, and a quantifiable dissipated work —
  all validated against the analytic limit.
* XPBD slides the block correctly (the slip curve agrees qualitatively) but provides
  **no calibrated friction force or dissipation** — the same limitation seen with the
  normal contact force in Stage B. Use the fast solver for plausible motion; use FEM
  when you need the force and the energy budget.